# Step 2B: Background Detection from the Image

Two ways to remove background exist in SPACEMAP. When the panel includes blank cycles,
subtracting them is the most direct route (Step 2A / the first Tab 2 subtab). This notebook
covers the other case: **no blank cycles**, so each marker's background is estimated from its
own pixel histogram.

## Why pixels and not cell means
A cell mean has already averaged noise over a few hundred pixels, which narrows and shifts the
background mode. A threshold derived from cell means does not mean the same thing as one derived
from pixels, so the estimate is made on the image.

## Why the tissue mask matters
Empty slide dominates an unmasked histogram and drags the estimate far too low. On one of our
ROIs, CD3e was estimated at **8** without a mask and **123** inside the annotated tissue - the
manually gated value was **104**.

## What this produces
- a threshold table in the exact CSV layout the **RESOLVE** phenotyping step reads
- optionally, a background-subtracted copy of the image

> **Reference-script switch:** the committed Groovy file currently has `useFullImage=true`, which
> bypasses annotations. Set it to `false` before comparing its tissue-annotation result with this notebook.

In [ ]:
import os, sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile

PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from spacemap.background_detection import (
    DEFAULT_MIN_COMPONENT_AREA_PX, THRESHOLD_METHODS, image_levels, find_background, detect_backgrounds,
    geojson_component_summary, mask_from_geojson, masks_from_geojson, otsu_tissue_mask,
    write_threshold_csv, threshold_csv_name, subtract_background_image,
)
from spacemap.tissue_annotation_utils import PyramidImageReader
print("threshold methods:", list(THRESHOLD_METHODS))

## 1. Configuration

In [ ]:
PROJECT   = PROJECT_ROOT / "project/INNATE_4_IMAGES_08052026"
IMAGE     = PROJECT / "outputs/rois/P2_P10_roi_4.ome.tiff"
ANNOTATION = PROJECT / "outputs/processed_annotations/P2_P10_roi_4_merged.geojson"
THRESHOLD_DIR = PROJECT / "data/thresholds"      # RESOLVE reads from here

METHOD          = "Triangle"     # ImageJ-compatible, as used by the Groovy script
TISSUE_CLASSES  = ["tissue"]     # which annotation classes count as tissue
DOWNSAMPLE      = 5              # matches thresholdDownsample=5 in the Groovy script
MIN_COMPONENT_AREA_PX = DEFAULT_MIN_COMPONENT_AREA_PX  # 30,000 full-resolution px²
MATCH_QUPATH_ANNOTATIONS = True  # threshold each GeoJSON feature, then take the mean
EXCLUDE_PREFIX  = "HE_"          # virtual H&E layers are not markers

stem = IMAGE.name.replace(".ome.tiff", "").replace(".ome.tif", "")
print("image:", IMAGE.name)

## 2. Image structure

The Groovy reference uses `thresholdDownsample=5`. Lower-resolution reads are much faster, but
averaging changes the pixel distribution, so another downsample will not return the same threshold.
The reader resamples the 4x pyramid to a true 5x grid rather than silently using 4x.

In [ ]:
levels = image_levels(IMAGE)
display(levels)

reader = PyramidImageReader(IMAGE)
markers = [c for c in reader.channel_names if not c.upper().startswith(EXCLUDE_PREFIX.upper())]
print(f"{len(reader.channel_names)} channels, {len(markers)} markers")

with tifffile.TiffFile(IMAGE) as tif:
    SOURCE_DTYPE = np.dtype(tif.pages[0].dtype)
print(f"measuring at {DOWNSAMPLE}x downsample: {reader.target_shape(DOWNSAMPLE)}; dtype={SOURCE_DTYPE}")

## 3. Tissue mask

In [ ]:
shape = reader.target_shape(DOWNSAMPLE)
if ANNOTATION.exists():
    component_summary = geojson_component_summary(
        ANNOTATION, TISSUE_CLASSES, MIN_COMPONENT_AREA_PX)
    mask = mask_from_geojson(ANNOTATION, shape, DOWNSAMPLE, classes=TISSUE_CLASSES,
                             min_component_area_px=MIN_COMPONENT_AREA_PX)
    source = f"annotation ({', '.join(TISSUE_CLASSES)})"
else:
    with tifffile.TiffFile(IMAGE) as tif:
        first = reader.read_channel(tif, 0, DOWNSAMPLE)
    mask = otsu_tissue_mask(first)
    source = "automatic Otsu fallback"
print(f"mask from {source}: {mask.sum():,} of {mask.size:,} pixels ({mask.mean():.1%})")
if ANNOTATION.exists():
    kept = component_summary['kept_components']; ignored = component_summary['ignored_components']
    retained = 100 * component_summary['kept_area_px'] / max(component_summary['total_area_px'], 1)
    print(f"components: {kept} kept, {ignored} below {MIN_COMPONENT_AREA_PX:,.0f} px² ignored; "
          f"{retained:.3f}% of annotation area retained")

plt.figure(figsize=(5, 5)); plt.imshow(mask, cmap="gray")
plt.title(f"tissue mask - {mask.mean():.1%} of the frame"); plt.axis("off"); plt.show()

## 4. Compare methods on one marker

The methods disagree substantially, so this is worth looking at before trusting any of them.
On our data **Triangle inside the tissue mask reproduced manually gated thresholds most closely**,
while the geometric elbow - which finds the shoulder of the noise mode - ran 3-5x lower.

In [ ]:
PROBE = "CD3e" if "CD3e" in markers else markers[0]
with tifffile.TiffFile(IMAGE) as tif:
    plane = reader.read_channel(tif, list(reader.channel_names).index(PROBE), DOWNSAMPLE)
pixels = plane[mask]
rows_in_mask = np.flatnonzero(mask.any(axis=1)); cols_in_mask = np.flatnonzero(mask.any(axis=0))
request_rectangle = plane[rows_in_mask[0]:rows_in_mask[-1]+1, cols_in_mask[0]:cols_in_mask[-1]+1]
imagej_range = (float(request_rectangle.min()), float(request_rectangle.max()))

rows = [{"method": m, "threshold": find_background(
    pixels, m, value_range=imagej_range, imagej_dtype=SOURCE_DTYPE)["threshold"]}
    for m in THRESHOLD_METHODS]
display(pd.DataFrame(rows).round(1))

result = find_background(pixels, METHOD, value_range=imagej_range, imagej_dtype=SOURCE_DTYPE)
upper = np.percentile(pixels, 99.5)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(pixels[pixels <= upper], bins=150, color="#7f8c8d")
ax[0].axvline(result["threshold"], color="#c0392b", ls="--", lw=2,
              label=f"{METHOD}: {result['threshold']:.1f}")
ax[0].set_yscale("log"); ax[0].legend(frameon=False)
ax[0].set_title(f"{PROBE} - pixels inside the mask"); ax[0].set_xlabel("intensity")
ax[1].imshow(np.clip(plane - result["threshold"], 0, None), cmap="magma", vmax=max(upper, 1))
ax[1].set_title(f"after subtracting {result['threshold']:.1f}"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 5. Detect every marker

`MATCH_QUPATH_ANNOTATIONS=True` reproduces the Groovy loop: each GeoJSON feature is one QuPath
annotation object, all retained disconnected pieces in that feature are pooled into one histogram,
and multiple annotation-object thresholds are combined with the mean.

Do not split a `MultiPolygon` into thousands of independent threshold samples. That was the GUI
discrepancy: tiny/constant fragments caused 150 fallbacks and a median unlike QuPath's result.

In [ ]:
if MATCH_QUPATH_ANNOTATIONS and ANNOTATION.exists():
    regions = masks_from_geojson(
        ANNOTATION, shape, DOWNSAMPLE, classes=TISSUE_CLASSES, region_unit="feature",
        min_component_area_px=MIN_COMPONENT_AREA_PX)
    print(f"{len(regions)} regions:", ", ".join(regions))
    table = detect_backgrounds(IMAGE, markers=markers, method=METHOD,
                               downsample=DOWNSAMPLE, region_masks=regions, aggregate="mean")
    if (table["region_spread"] > 3).any():
        worst = table.loc[table["region_spread"].idxmax()]
        print(f"WARNING: up to {worst['region_spread']:.0f}x spread between regions "
              f"(worst: {worst['Marker']}) - check whether these are compartments, not background samples")
else:
    table = detect_backgrounds(IMAGE, markers=markers, method=METHOD,
                               downsample=DOWNSAMPLE, mask=mask,
                               progress=lambda d, t, n: print(f"  {n} ({d}/{t})", end="\r"))

display(table[["Marker", "threshold", "peak", "n_pixels"]].round(1))

## 6. Review, and override anything that looks wrong

In [ ]:
OVERRIDES = {
    # "CD4": 900.0,
}
table["manual_override"] = table["Marker"].map(OVERRIDES)
table["final"] = table["manual_override"].fillna(table["threshold"])
changed = table.loc[table["manual_override"].notna(), ["Marker", "threshold", "final"]]
print(f"{len(changed)} manual override(s)")
display(changed if len(changed) else table[["Marker", "final"]].round(1).head(10))

## 7. Save the threshold table

Written in the layout RESOLVE searches for first (`<stem>.ome.tiff_-_Merged_Image.csv`), with the
value in `Polygon_1`. Annotation-object thresholds are averaged before export, matching the
Groovy script rather than asking RESOLVE to take a different statistic later.

In [ ]:
export = table.copy()
export["threshold"] = export["final"]
path = write_threshold_csv(export, THRESHOLD_DIR / threshold_csv_name(stem), export_mode="mean")
print("saved:", path)

check = pd.read_csv(path)
polygon_cols = [c for c in check.columns if c.startswith("Polygon_")]
print(f"{len(polygon_cols)} Polygon_* column(s); RESOLVE will use:")
display(check.set_index("Marker")[polygon_cols].round(1).head(10))

## 8. Optional: write a background-subtracted image

Full resolution, streamed channel by channel, storage dtype and channel names preserved. This is
slow on multi-gigabyte ROIs - run one image first and time it before starting a batch.

In [ ]:
WRITE_IMAGE = False   # set True to run

if WRITE_IMAGE:
    out_dir = PROJECT / "outputs/background_subtracted"
    target = out_dir / f"{stem}_bg_detected.ome.tiff"
    subtract_background_image(
        IMAGE, dict(zip(table["Marker"], table["final"].astype(float))), target,
        progress=lambda d, t, n: print(f"  {n} ({d}/{t})", end="\r"),
    )
    print("\nwritten:", target)
else:
    print("set WRITE_IMAGE = True to export")

## 9. Batch

Same settings applied to every image in the folder. Each needs its own annotation GeoJSON;
images without one fall back to the automatic Otsu mask, which is less reliable.

In [ ]:
RUN_BATCH = False

if RUN_BATCH:
    rows = []
    for image in sorted((PROJECT / "outputs/rois").glob("*.ome.tiff")):
        s = image.name.replace(".ome.tiff", "")
        annotation = PROJECT / "outputs/processed_annotations" / f"{s}_merged.geojson"
        rdr = PyramidImageReader(image)
        ds = DOWNSAMPLE
        shp = rdr.target_shape(ds)
        mk = [c for c in rdr.channel_names if not c.upper().startswith(EXCLUDE_PREFIX.upper())]
        regions = masks_from_geojson(
            annotation, shp, ds, classes=TISSUE_CLASSES, region_unit="feature",
            min_component_area_px=MIN_COMPONENT_AREA_PX) if annotation.exists() else None
        t = detect_backgrounds(image, markers=mk, method=METHOD, downsample=ds,
                               region_masks=regions, aggregate="mean")
        write_threshold_csv(t, THRESHOLD_DIR / threshold_csv_name(s), export_mode="mean")
        rows.append({"image": s, "markers": len(t), "downsample": ds,
                     "mask": "annotation" if regions is not None else "none",
                     "median threshold": round(float(t["threshold"].median()), 1)})
        print(f"  {s}: done")
    display(pd.DataFrame(rows))
else:
    print("set RUN_BATCH = True to process every image")

## Summary

1. Background is estimated per marker inside QuPath-equivalent annotation objects
2. ImageJ Triangle and 5x downsample match the Groovy reference defaults
3. Disconnected pieces below 30,000 full-resolution px² are ignored by default
4. Annotation thresholds are averaged and exported directly for RESOLVE
5. Optionally the correction can be baked into a copy of the image

**Keep the resolution level consistent** across images you intend to compare, and re-run RESOLVE
after regenerating thresholds - phenotype calls will shift.